# Praktikum Data Mining: Analisis Data Video CCTV ATCS Kota Semarang
## Milestone 1: Data Acquisition, Profiling & Preprocessing Pipeline

**Topik Penelitian:** Pemetaan Pola Spasial-Temporal Kepadatan Lalu Lintas Kota Semarang Berbasis Ekstraksi Objek Kendaraan dari Rekaman CCTV ATCS Publik Menggunakan YOLOv8 dan K-Means Clustering  
**Dataset Sumber:** Stream Publik HLS Dishub Kota Semarang (*Pantau Semar* - `pantausemar.semarangkota.go.id`)  
**Cakupan Geografis:** 12 Titik Persimpangan Strategis (Barat, Pusat/CBD, Timur, Selatan)  
**Fase Saat Ini:** Ekstraksi Metadata Video & Preprocessing Data Mentah (Sesuai Panduan CRISP-DM)

---
### Alur Preprocessing yang Diterapkan:
1. **Profiling Data Mentah**: Menelaah struktur rekaman, resolusi citra, dan status stream.
2. **Data Cleaning**: Menangani anomali *stream error* (kamera offline) dan duplikasi frame (*stream freeze* via SHA-256 & dHash).
3. **Evaluasi Kualitas Citra**: Profil kecerahan (`mean_luminance`) dan tingkat ketajaman/blur (`laplacian_variance`).
4. **Penanganan Missing Value**: Imputasi dan audit keteraturan timestamp capture.
5. **Normalisasi Numerik**: Standarisasi metrik citra menggunakan MinMaxScaler.
6. **Encoding & Feature Engineering**: Penambahan fitur temporal (rush hour, segmen waktu) dan spasial (zonasi koridor).
7. **Analisis Statistik & Outlier**: Perhitungan Mean, Variance, Std Dev, serta deteksi outlier dengan metode Interquartile Range (IQR).
8. **Export Dataset Siap Modeling**: Menyimpan dataset bersih untuk tahap inferensi YOLOv8 dan K-Means Clustering.

### 1. Inisialisasi Environment & Import Library

In [ ]:
# Library standar data science & computer vision (semuanya pre-installed di Google Colab)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from scipy import stats
import folium
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110
print("Environment siap digunakan!")

### 2. Memuat Data Mentah & Registry Kamera
Data yang dimuat terdiri dari:
- `raw_cctv_metadata.csv`: Catatan rekaman riil dari penangkap stream multi-kamera Semarang.
- `cameras.csv`: Registri geospasial (koordinat lat-lng dan zonasi koridor) dari 12 persimpangan.

In [ ]:
# Load dataset mentah dan registry kamera
meta_path = 'data/raw_cctv_metadata.csv' if os.path.exists('data/raw_cctv_metadata.csv') else 'raw_cctv_metadata.csv'
cam_path = 'data/cameras.csv' if os.path.exists('data/cameras.csv') else 'cameras.csv'

df_raw = pd.read_csv(meta_path)
df_cams = pd.read_csv(cam_path)

print(f"Total Baris Metadata Mentah : {len(df_raw)}")
print(f"Total Kamera Terdaftar       : {len(df_cams)}")
display(df_cams[['camera_id', 'camera_name', 'zone', 'latitude', 'longitude']])
display(df_raw.head())

### 3. Profiling Data Mentah (Data Understanding)
Memeriksa distribusi status capture, resolusi video stream, dan kelengkapan data sebelum dibersihkan.

In [ ]:
# Ringkasan status capture
status_counts = df_raw['capture_status'].value_counts()
print("Distribusi Status Capture:")
print(status_counts)

# Cek resolusi per kamera
res_summary = df_raw[df_raw['capture_status'] == 'captured'].groupby('camera_name')[['resolution_w', 'resolution_h']].first()
print("\nResolusi Video Per Kamera ATCS:")
display(res_summary)

# Visualisasi status capture mentah
plt.figure(figsize=(7, 3))
sns.countplot(data=df_raw, x='capture_status', palette='Set2')
plt.title('Distribusi Status Tangkapan Stream CCTV Mentah')
plt.xlabel('Status')
plt.ylabel('Jumlah Baris')
plt.tight_layout()
plt.show()

### 4. Data Cleaning (Pembersihan Data)
Tahap ini menangani anomali riil yang terjadi pada stream CCTV publik:
1. **Pemisahan Data Anomali/Offline**: Baris dengan status `stream_error` dicatat sebagai histori *downtime*, sementara frame valid dipisahkan untuk analisis citra.
2. **Deteksi Duplikasi (Stream Freeze)**: Jika terjadi *stream freeze*, hash citra (`sha256` atau `dhash_hex`) akan identik antar frame berurutan.
3. **Tipe Data Parsing**: Mengonversi `capture_timestamp` menjadi tipe `datetime` lokal (WIB).

In [ ]:
# Salin dataframe untuk proses cleaning
df_clean = df_raw.copy()

# 4.1 Parsing Timestamp
df_clean['capture_timestamp'] = pd.to_datetime(df_clean['capture_timestamp'], errors='coerce')

# 4.2 Identifikasi & Penanganan Missing / Offline Data
total_attempts = len(df_clean)
offline_records = df_clean[df_clean['capture_status'] != 'captured']
print(f"Tingkat Keberhasilan Capture: {((total_attempts - len(offline_records)) / total_attempts * 100):.1f}%")
print(f"Jumlah Record Offline / Error: {len(offline_records)}")

# 4.3 Filter hanya frame yang berhasil ditangkap untuk analisis visual
df_valid = df_clean[df_clean['capture_status'] == 'captured'].copy()

# Konversi metrik citra ke tipe numerik
num_cols = ['resolution_w', 'resolution_h', 'mean_luminance', 'laplacian_variance']
for col in num_cols:
    df_valid[col] = pd.to_numeric(df_valid[col], errors='coerce')

# 4.4 Identifikasi Frame Beku / Duplikat (Stream Freeze)
dup_count = df_valid['is_duplicate_of_previous'].astype(str).str.lower().isin(['true', '1']).sum()
print(f"Jumlah Frame Duplikat (Stream Freeze) Terdeteksi: {dup_count}")

print(f"Total frame valid siap dianalisis: {len(df_valid)}")

### 5. Analisis Kualitas Citra (Brightness & Blur Profiling)
Sesuai rumusan masalah jurnal, kondisi cuaca, pencahayaan, dan kebersihan lensa CCTV bervariasi:
- `mean_luminance`: Kecerahan rata-rata (skala 0–255). Nilai < 40 mengindikasikan malam/gelap gulita atau kamera tertutup, nilai > 220 mengindikasikan *glare* / silau lampu.
- `laplacian_variance`: Ketajaman citra. Nilai rendah (< 100) mengindikasikan gambar buram/kabur (out of focus, hujan deras, lensa kotor).

In [ ]:
# Distribusi Luminance & Laplacian Variance
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(df_valid['mean_luminance'], kde=True, ax=axes[0], color='goldenrod')
axes[0].set_title('Distribusi Kecerahan Citra (Mean Luminance)')
axes[0].set_xlabel('Mean Luminance (0-255)')
axes[0].axvline(df_valid['mean_luminance'].mean(), color='red', linestyle='--', label=f"Mean: {df_valid['mean_luminance'].mean():.1f}")
axes[0].legend()

sns.histplot(df_valid['laplacian_variance'], kde=True, ax=axes[1], color='teal')
axes[1].set_title('Distribusi Ketajaman Citra (Laplacian Variance)')
axes[1].set_xlabel('Laplacian Variance (Ketajaman)')
axes[1].axvline(df_valid['laplacian_variance'].median(), color='red', linestyle='--', label=f"Median: {df_valid['laplacian_variance'].median():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

### 6. Analisis Statistik Deskriptif (Sesuai Panduan Tugas Praktikum Dosen)
Menghitung **Mean, Variance, Standar Deviasi** metrik citra dan performa stream per kamera, serta mendeteksi **Outlier** menggunakan metode **Interquartile Range (IQR)**.

In [ ]:
# 6.1 Tabel Statistik Deskriptif per Kamera
stat_table = df_valid.groupby(['camera_id', 'camera_name']).agg({
    'mean_luminance': ['count', 'mean', 'var', 'std'],
    'laplacian_variance': ['mean', 'var', 'std']
})
stat_table.columns = ['Count', 'Lum_Mean', 'Lum_Var', 'Lum_Std', 'Sharp_Mean', 'Sharp_Var', 'Sharp_Std']
print("Tabel Statistik Deskriptif Kualitas Citra CCTV per Kamera:")
display(stat_table.round(2))

# 6.2 Deteksi Outlier dengan Metode IQR (Interquartile Range)
def detect_iqr_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return (series < lower_bound) | (series > upper_bound)

df_valid['is_luminance_outlier'] = detect_iqr_outliers(df_valid['mean_luminance'])
df_valid['is_sharpness_outlier'] = detect_iqr_outliers(df_valid['laplacian_variance'])

print(f"Outlier Kecerahan (IQR): {df_valid['is_luminance_outlier'].sum()} frame")
print(f"Outlier Ketajaman (IQR): {df_valid['is_sharpness_outlier'].sum()} frame")

### 7. Feature Engineering & Penggabungan Spasial-Temporal
1. **Spasial Join**: Menggabungkan data frame dengan koordinat (`latitude`, `longitude`) dan zonasi kota dari `cameras.csv`.
2. **Fitur Temporal**: Ekstraksi jam, menit, status *rush hour* (jam sibuk pagi 06:30–08:30 WIB).
3. **Normalisasi (MinMaxScaler)**: Menormalisasi fitur numerik ke rentang [0, 1] agar siap diolah oleh algoritma clustering.

In [ ]:
# 7.1 Gabungkan dengan Registry Kamera (Spasial Join)
df_enriched = df_valid.merge(
    df_cams[['camera_id', 'zone', 'latitude', 'longitude']], 
    on='camera_id', 
    how='left'
)

# 7.2 Ekstraksi Fitur Temporal
df_enriched['hour'] = df_enriched['capture_timestamp'].dt.hour
df_enriched['minute'] = df_enriched['capture_timestamp'].dt.minute
df_enriched['time_str'] = df_enriched['capture_timestamp'].dt.strftime('%H:%M')
# Identifikasi Jam Padat / Rush Hour (06:30 - 08:30 WIB)
df_enriched['is_rush_hour'] = df_enriched['capture_timestamp'].apply(
    lambda ts: 1 if (ts.hour == 6 and ts.minute >= 30) or (ts.hour == 7) or (ts.hour == 8 and ts.minute <= 30) else 0
)

# 7.3 Normalisasi Fitur Numerik menggunakan MinMaxScaler
scaler = MinMaxScaler()
norm_cols = ['mean_luminance', 'laplacian_variance']
norm_feature_names = [f'{c}_scaled' for c in norm_cols]
df_enriched[norm_feature_names] = scaler.fit_transform(df_enriched[norm_cols])

# 7.4 Categorical Encoding untuk Zona Spasial
le = LabelEncoder()
df_enriched['zone_encoded'] = le.fit_transform(df_enriched['zone'])

print("Tampilan Data Setelah Feature Engineering & Normalisasi:")
display(df_enriched[['camera_id', 'camera_name', 'zone', 'is_rush_hour', 'mean_luminance_scaled', 'laplacian_variance_scaled', 'zone_encoded']].head())

### 8. Visualisasi Spasial & Boxplot Distribusi (Sesuai Luaran Praktikum)
Menampilkan perbandingan kualitas citra antar-lokasi dan peta interaktif sebaran 12 kamera ATCS di Kota Semarang.

In [ ]:
# 8.1 Boxplot Kecerahan per Persimpangan
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_enriched, x='camera_name', y='mean_luminance', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Boxplot Distribusi Kecerahan Citra (Mean Luminance) Antar Persimpangan CCTV')
plt.xlabel('Nama Persimpangan')
plt.ylabel('Kecerahan')
plt.tight_layout()
plt.show()

# 8.2 Boxplot Ketajaman per Zona Wilayah
plt.figure(figsize=(11, 5))
sns.boxplot(data=df_enriched, x='zone', y='laplacian_variance', palette='coolwarm')
plt.xticks(rotation=40, ha='right')
plt.title('Variansi Ketajaman Citra Berdasarkan Zona Wilayah Kota Semarang')
plt.xlabel('Zona Wilayah')
plt.ylabel('Ketajaman (Laplacian Variance)')
plt.tight_layout()
plt.show()

### 9. Peta Sebaran Spasial 12 Titik Kamera ATCS Kota Semarang
Peta interaktif berikut memetakan ke-12 kamera CCTV di atas peta topografi jalan Kota Semarang menggunakan pustaka `folium` dengan provider peta **Google Maps** (bebas API key, bebas watermark, dan memuat nama jalan persimpangan Semarang dengan jelas).

In [ ]:
# Titik pusat Kota Semarang dengan Google Maps Tiles (Bebas Watermark & Tanpa API Key)
semarang_map = folium.Map(
    location=[-6.9932, 110.4203], 
    zoom_start=13, 
    tiles='https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
    attr='Google Maps'
)

for idx, cam in df_cams.iterrows():
    folium.Marker(
        location=[cam['latitude'], cam['longitude']],
        popup=f"<b>{cam['camera_id']} - {cam['camera_name']}</b><br>Zona: {cam['zone']}",
        tooltip=f"{cam['camera_id']}: {cam['camera_name']}",
        icon=folium.Icon(color='red' if 'Pusat' in str(cam['zone']) else 'blue', icon='info-sign')
    ).add_to(semarang_map)

# Tampilkan peta interaktif
semarang_map

### 10. Ekspor Dataset Bersih (Siap Tahap Modeling YOLOv8 & K-Means)
Dataset yang sudah melalui tahap pembersihan, penanganan data offline, normalisasi, dan pengayaan fitur disimpan sebagai basis inferensi model.

In [ ]:
# Ekspor ke CSV bersih
output_clean_csv = 'data/clean_cctv_metadata.csv'
df_enriched.to_csv(output_clean_csv, index=False)
print(f"[SUKSES] Dataset hasil preprocessing berhasil disimpan ke: {output_clean_csv}")
print(f"Ukuran data bersih: {df_enriched.shape[0]} baris x {df_enriched.shape[1]} kolom.")

# Preview ringkasan kolom siap modeling
print("\nKolom Siap Analisis / Modeling:")
print(list(df_enriched.columns))

---
### 11. Kesimpulan Milestone Preprocessing & Rencana Tahap Selanjutnya

1. **Capaian Milestone Saat Ini**:
   - Berhasil membangun pipeline penangkapan streaming multi-kamera dari 12 persimpangan utama Kota Semarang secara *real-time*.
   - Data mentah telah di-profiling dan dibersihkan dari kendala jaringan (*stream error* dan *freeze frame*).
   - Metrik kualitas citra telah diuji normalitas dan distribusinya, serta dihitung nilai Mean, Variance, dan Standar Deviasi per kamera.
   - Data spasial (koordinat geospasial) dan temporal (jam sibuk) berhasil diintegrasikan.

2. **Rencana Tahap Modeling (Milestone 2)**:
   - **Ekstraksi Objek Kendaraan**: Menjalankan model `YOLOv8` (COCO pre-trained) pada frame bersih untuk mendeteksi `car`, `motorcycle`, `bus`, dan `truck`.
   - **Agregasi Fitur Lalu Lintas**: Menghitung kepadatan kendaraan per jam per persimpangan.
   - **Clustering**: Menerapkan algoritma `K-Means` / `DBSCAN` untuk mengelompokkan persimpangan ke dalam kluster *Tingkat Kepadatan / Risiko Kemacetan* (*High Density/Risk* vs *Low Density/Risk*).
   - **Visualisasi Akhir**: Peta spasial-temporal *traffic density* interaktif berbasis web/GIS.